*Imports & Basic Paths*

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from glob import glob
import os
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd

DATA_ROOT = Path(r"D:\Dataset")
YAML_PATH = DATA_ROOT / "plantdoc.yaml"
WEIGHTS_PATH = DATA_ROOT / "yolov8n.pt"

print("DATA_ROOT:", DATA_ROOT)
print("YAML_PATH:", YAML_PATH)
print("WEIGHTS_PATH:", WEIGHTS_PATH)

print("Train images dir:", DATA_ROOT / "train" / "images")
print("Test images dir:", DATA_ROOT / "test" / "images")

*Detect Number of Classes*

In [ ]:
labels_root_train = DATA_ROOT / "train" / "labels"
labels_root_test = DATA_ROOT / "test" / "labels"

label_files = glob(str(labels_root_train / "*.txt")) + \
              glob(str(labels_root_test / "*.txt"))

print("Total label files found:", len(label_files))
print("First few label files:")
for lf in label_files[:5]:
    print("  ", lf)

cls_ids = set()

for lf in label_files:
    with open(lf, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            try:
                cls = int(parts[0])
                cls_ids.add(cls)
            except ValueError:
                pass

print("\nUnique class ids:", sorted(cls_ids))

if not cls_ids:
    raise RuntimeError("No Class ID Found!")

max_cls = max(cls_ids)
nc = max_cls + 1
print("Max class id:", max_cls)
print("=> nc should be:", nc)

*Update plantdoc.yaml*

In [ ]:
cfg = {
    "path": "D:/Dataset",
    "train": "train/images",
    "val": "test/images",
    "test": "test/images",
    "nc": int(nc),
    "names": [f"class_{i}" for i in range(int(nc))],
}

print("Saving YAML to:", YAML_PATH)

with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

print("✅ plantdoc.yaml written.")
print(cfg)

*Check Paths & Load Pretrained Model*

In [ ]:
with open(YAML_PATH, "r", encoding="utf-8") as f:
    cfg_loaded = yaml.safe_load(f)

print("Loaded from YAML:")
print(cfg_loaded)

root = cfg_loaded["path"]
train_dir = os.path.join(root, cfg_loaded["train"])
val_dir = os.path.join(root, cfg_loaded["val"])

print("\nTrain dir:", train_dir, "exists:", os.path.exists(train_dir))
print("Val dir:", val_dir, "exists:", os.path.exists(val_dir))

assert WEIGHTS_PATH.exists(), f"Weights file not found: {WEIGHTS_PATH}"

model = YOLO(str(WEIGHTS_PATH))
print("\n✅ Pretrained model loaded:", WEIGHTS_PATH)


*Run Models*

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml
import os

DATA_ROOT = Path(r"D:\Dataset")
YAML_PATH = DATA_ROOT / "plantdoc.yaml"
WEIGHTS_PATH = DATA_ROOT / "yolov8n.pt"

model = YOLO(str(WEIGHTS_PATH))

results = model.train(
    data=str(YAML_PATH),
    epochs=400,
    imgsz=416,
    batch=8,
    device=0,
    amp=False,
    workers=0,
    close_mosaic=0,
    lr0=1e-3,
    weight_decay=5e-4,
    project=str(DATA_ROOT / "runs_final2"),
    name="yolov8n_plantdoc",
    pretrained=True,
    patience=10,
    verbose=True,
)

print("✅ Training finished.")

*Functions for GT vs Pred Visualization*

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from glob import glob
import os
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd

%matplotlib inline

DATA_ROOT = Path(r"D:\Dataset")
YAML_PATH = DATA_ROOT / "plantdoc.yaml"
BEST_WEIGHTS = DATA_ROOT / "runs_final2" / "yolov8n_plantdoc" / "weights" / "best.pt"

print("DATA_ROOT:", DATA_ROOT)
print("YAML_PATH:", YAML_PATH)
print("BEST_WEIGHTS:", BEST_WEIGHTS)

assert YAML_PATH.exists(), f"YAML not found: {YAML_PATH}"
assert BEST_WEIGHTS.exists(), f"best.pt not found: {BEST_WEIGHTS}"

# YAML → گرفتن نام کلاس‌ها
with open(YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg.get("names", None)
print("nc:", data_cfg.get("nc"))
print("len(names):", len(class_names) if class_names else None)

best_model = YOLO(str(BEST_WEIGHTS))
print("✅ Best model loaded.")

*Plot All Losses & Metrics from results.csv*

In [ ]:
run_dir = DATA_ROOT / "runs_final2" / "yolov8n_plantdoc"
results_csv = run_dir / "results.csv"

print("Results path:", results_csv)
assert results_csv.exists(), f"results.csv not found at: {results_csv}"

df = pd.read_csv(results_csv)
print("Columns:")
print(df.columns.tolist())

epochs = df["epoch"]

train_loss_cols = [c for c in df.columns if c.startswith("train/") and "loss" in c]
val_loss_cols   = [c for c in df.columns if c.startswith("val/")   and "loss" in c]

print("\nTrain loss columns:", train_loss_cols)
print("Val loss columns:", val_loss_cols)

plt.figure(figsize=(10, 6))
for col in train_loss_cols:
    plt.plot(epochs, df[col], label=col)

for col in val_loss_cols:
    plt.plot(epochs, df[col], linestyle="--", label=col)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Losses")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

metric_cols = [c for c in df.columns if c.startswith("metrics/")]
print("\nMetric columns:", metric_cols)

plt.figure(figsize=(10, 6))
for col in metric_cols:
    plt.plot(epochs, df[col], label=col)

plt.xlabel("Epoch")
plt.ylabel("Metric value")
plt.title("Validation Metrics (precision, recall, mAP)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

best_path = Path(r"D:\Dataset\runs_final2\yolov8n_plantdoc\weights\best.pt")
last_path = Path(r"D:\Dataset\runs_final2\yolov8n_plantdoc\weights\last.pt")

print("best.pt exists:", best_path.exists(), "->", best_path)
print("last.pt exists:", last_path.exists(), "->", last_path)


In [ ]:
def load_yolo_labels(label_path, img_width, img_height):

    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id = int(parts[0])
            x_c, y_c, w, h = map(float, parts[1:])

            x_c *= img_width
            y_c *= img_height
            w   *= img_width
            h   *= img_height

            x1 = x_c - w / 2
            y1 = y_c - h / 2
            x2 = x_c + w / 2
            y2 = y_c + h / 2

            boxes.append({
                "cls": cls_id,
                "bbox": [x1, y1, x2, y2]
            })
    return boxes


def plot_gt_vs_pred(img_path, label_path, model, class_names=None,
                    conf_th=0.25, iou_th=0.5, device=0):

    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)
    h, w = img_np.shape[:2]

    gt_boxes = load_yolo_labels(label_path, w, h)

    result = model.predict(
        source=str(img_path),
        imgsz=416,
        conf=conf_th,
        iou=iou_th,
        device=device,
        verbose=False
    )[0]

    pred_boxes = []
    if result.boxes is not None:
        for b in result.boxes:
            cls_id = int(b.cls.item())
            x1, y1, x2, y2 = b.xyxy[0].cpu().numpy().tolist()
            score = float(b.conf.item())
            pred_boxes.append({
                "cls": cls_id,
                "bbox": [x1, y1, x2, y2],
                "score": score
            })

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img_np)
    ax.set_axis_off()

    for gt in gt_boxes:
        x1, y1, x2, y2 = gt["bbox"]
        cls_id = gt["cls"]
        label = f"GT {class_names[cls_id]}" if class_names else f"GT {cls_id}"
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             fill=False, edgecolor="green", linewidth=2)
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 2, label,
            fontsize=8, color="white",
            bbox=dict(facecolor="green", alpha=0.6, edgecolor="none")
        )

    for pr in pred_boxes:
        x1, y1, x2, y2 = pr["bbox"]
        cls_id = pr["cls"]
        score = pr["score"]
        label = f"Pred {class_names[cls_id]} {score:.2f}" if class_names else f"Pred {cls_id} {score:.2f}"
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             fill=False, edgecolor="red", linewidth=2)
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 2, label,
            fontsize=8, color="white",
            bbox=dict(facecolor="red", alpha=0.6, edgecolor="none")
        )

    plt.tight_layout()
    plt.show()

In [ ]:
test_images_dir = DATA_ROOT / "test" / "images"
test_labels_dir = DATA_ROOT / "test" / "labels"

num_samples = 40
sample_imgs = glob(str(test_images_dir / "*.jpg"))[:num_samples]

print("Selected images:")
for img_path in sample_imgs:
    print("  ", img_path)

for img_path in sample_imgs:
    img_path = Path(img_path)
    label_path = test_labels_dir / (img_path.stem + ".txt")
    print("\nImage:", img_path.name)
    print("Label:", label_path.name)
    plot_gt_vs_pred(
        img_path=img_path,
        label_path=label_path,
        model=best_model,
        class_names=class_names,
        device=0
    )